In [1]:
import polars as pl
from sentence_transformers import SentenceTransformer
import numpy as np
import json
from google.colab import drive
import os

EMBEDDING_MODEL = "ibm-granite/granite-embedding-30m-english"

In [2]:
drive.mount('/content/drive') # Mount Google Drive to access data files

Mounted at /content/drive


In [3]:
# Load filtered book parquet from Google Drive
books_df = pl.read_parquet('/content/drive/MyDrive/books_with_genres.parquet')

In [4]:
print(len(books_df))

1782579


In [5]:
# Load embedding model
model = SentenceTransformer(EMBEDDING_MODEL, device = "cuda")

# half model size for faster inference and reduced memory usage, this is a feature of the ibm-granite models that they can be loaded in half precision which reduces memory usage and speeds up inference without significant loss in embedding quality
model.half()

# make embedding directory if it doesn't exist
os.makedirs("/content/drive/MyDrive/embeddings", exist_ok=True)


# Generate embeddings for book descriptions + title concatenated, do this in batches to avoid memory issues
BATCH_SIZE = 256 # Adjust batch size based on available memory
embeddings = [] # Store embeddings in a list to avoid memory issues

# Create rows of books with concatenated title and description - handle nulls with list comprehension
books_df = books_df.with_columns(
    pl.when(pl.col("description").is_null() | (pl.col("description")== ""))
    .then(pl.col("title"))
    .otherwise(pl.col("title") + " - " + pl.col("description"))
    .alias("text")
)

SHARD_SIZE = 50000 # Process books in shards to avoid memory issues, adjust based on available memory
num_books = len(books_df)
for shard_idx in range(0, num_books, SHARD_SIZE):
    shard_path = f"/content/drive/MyDrive/embeddings/shard_{shard_idx:08d}.npy"
    if os.path.exists(shard_path):
        print(f"Shard {shard_idx} already exists, skipping...")
        continue
    shard_books = books_df[shard_idx:shard_idx+SHARD_SIZE]
    shard_texts = shard_books["text"].to_list()
    shard_embeddings = model.encode(
        shard_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True, # Normalize embeddings to unit length for cosine similarity
        convert_to_numpy=True # Convert to numpy array directly to save memory
    )
    np.save(shard_path, shard_embeddings)
    print(f"Saved shard {shard_idx} to {shard_path}")

# after all shards done concatenate
all_shard = [np.load(f"/content/drive/MyDrive/embeddings/{file}") for file in sorted(os.listdir("/content/drive/MyDrive/embeddings")) if file.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)


# Save matrix of embeddings as .npy file back to drive
np.save('/content/drive/MyDrive/book_embeddings.npy', embeddings)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Saved shard 0 to /content/drive/MyDrive/embeddings/shard_00000000.npy


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

...

Saved shard 1650000 to /content/drive/MyDrive/embeddings/shard_01650000.npy


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Saved shard 1700000 to /content/drive/MyDrive/embeddings/shard_01700000.npy


Batches:   0%|          | 0/128 [00:00<?, ?it/s]

Saved shard 1750000 to /content/drive/MyDrive/embeddings/shard_01750000.npy


In [6]:
# Save book_id to row index mapping as a json file to drive, this will be used later to map from book_id to embedding index when we want to retrieve embeddings for a given book_id
book_id_to_index = {row["book_id"]: idx for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open('/content/drive/MyDrive/book_id_to_index.json', 'w') as file:
    json.dump(book_id_to_index, file, indent=2)

# Build the reverse mapping from row index to book_id, this will be used later to map from embedding index back to book_id when we want to retrieve metadata for a given embedding
index_to_book_id = {idx: row["book_id"] for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open('/content/drive/MyDrive/index_to_book_id.json', 'w') as file:
    json.dump(index_to_book_id, file, indent=2)

In [ ]:
# Load the book_id to index from drive then save to local folder transformed/ to be used in the next steps of the pipeline, this is just to verify that the json file was saved correctly and can be loaded back without issues
with open('/content/drive/MyDrive/book_id_to_index.json', 'r') as file:
    book_id_to_index = json.load(file)
json.dump(book_id_to_index, open("data/transformed/book_id_to_index.json", "w"), indent=2)


# Load and concatenate all shard embeddings to verify they were saved correctly, then save to transformed/
all_shard = [np.load(f"/content/drive/MyDrive/embeddings/{file}") for file in sorted(os.listdir("/content/drive/MyDrive/embeddings")) if file.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)
np.save("data/transformed/embeddings.npy", embeddings)